In [1]:
from ROOT import TFile
import numpy as np
from itertools import product
from math import sqrt, log

Welcome to JupyROOT 6.26/08


In [2]:
HOME = "/home/choij/workspace/ChargedHiggsAnalysis"
ERAs = ["2016preVFP", "2016postVFP", "2017", "2018"]
SIGNALs = ["MHc-70_MA-15", "MHc-70_MA-40", "MHc-70_MA-65",
           "MHc-100_MA-15", "MHc-100_MA-60", "MHc-100_MA-95",
           "MHc-130_MA-15", "MHc-130_MA-55", "MHc-130_MA-90", "MHc-130_MA-125",
           "MHc-160_MA-15", "MHc-160_MA-85", "MHc-160_MA-120", "MHc-160_MA-155"]
NONPROMPTs = ["DYJets", "DYJets10to50_MG", "TTLL_powheg"]
VVs = ["WZTo3LNu_amcatnlo", "ZZTo4L_powheg"]
TTXs = ["ttWToLNu", "ttZToLLNuNu", "ttHToNonbb"]
CONVs = ["ZGToLLG", "TTG"]
RAREs = ["WWW", "WWZ", "WZZ", "ZZZ", "GluGluHToZZTo4L", "VBF_HToZZTo4L"]
BACKGROUNDs = NONPROMPTs+VVs+TTXs+CONVs+RAREs

In [11]:
CHANNEL = "SR1E2Mu"
MASSPOINT = "MHc-100_MA-60"
histkey = f"{CHANNEL}/{MASSPOINT}/3D"

# get signal histogram
f = TFile.Open(f"{HOME}/ValidParticleNet/Signals/TTToHcToWAToMuMu_{MASSPOINT}.root")
h_sig = f.Get(histkey); h_sig.SetDirectory(0)
f.Close()

# backgrounds
hists = {}
for bkg in BACKGROUNDs:
    f = TFile.Open(f"{HOME}/ValidParticleNet/Backgrounds/{bkg}.root")
    h_bkg = f.Get(histkey); h_bkg.SetDirectory(0)
    f.Close()
    hists[bkg] = h_bkg

In [12]:
def getNumberOfEvts(mA, cuts):
    mAcut, fcut, xcut = cuts
    xL, xR = h_sig.GetXaxis().FindBin(mA-mAcut), h_sig.GetXaxis().FindBin(mA+mAcut)
    yL, yR = h_sig.GetYaxis().FindBin(fcut), h_sig.GetYaxis().FindBin(1.)
    zL, zR = h_sig.GetZaxis().FindBin(xcut), h_sig.GetZaxis().FindBin(1.)
    
    # signal
    nSig = h_sig.Integral(xL, xR, yL, yR, zL, zR)
    
    # background
    nBkg = 0.
    for bkg in BACKGROUNDs:
        thisBkg = hists[bkg].Integral(xL, xR, yL, yR, zL, zR)
        if thisBkg < 0.:
            #print(f"negative bkgs for {bkg}")
            continue
        else:
            nBkg += thisBkg
    return (nSig, nBkg)

def getMetric(nSig, nBkg):
    return sqrt(2*((nSig+nBkg)*log(1+nSig/nBkg) - nSig))

In [13]:
nSig, nBkg = getNumberOfEvts(mA=60., cuts=[2., 0., 0.])
print(getMetric(nSig, nBkg))

17.13112030514609


In [14]:
mAcuts = np.linspace(0, 5, 51)
fcuts = np.linspace(0, 1, 21)
xcuts = np.linspace(0, 1, 21)

# 1D
metric = -1.
bestMAcut = 0.
for mAcut in mAcuts:
    nSig, nBkg = getNumberOfEvts(mA=60, cuts=[mAcut, 0., 0.])
    thisMetric = getMetric(nSig, nBkg)
    if thisMetric > metric: 
        metric = thisMetric
        bestMAcut = mAcut
print(f"1D Optimization: {metric} with best MA cut {bestMAcut}")

metric = -1.
bestMAcut = 0.
bestFcut = 0.
bestXcut = 0.
for mAcut, fcut, xcut in product(mAcuts, xcuts, fcuts):
    nSig, nBkg = getNumberOfEvts(mA=60, cuts=[mAcut, fcut, xcut])
    thisMetric = getMetric(nSig, nBkg)
    if thisMetric > metric: 
        metric = thisMetric
        bestMAcut = mAcut
        bestFcut = fcut
        bestXcut = xcut
print(f"3D Optimization: {metric} with best MA cut {bestMAcut}, fcut {bestFcut}, xcut {bestXcut}")

1D Optimization: 17.8674875902432 with best MA cut 0.9
3D Optimization: 21.88415778024672 with best MA cut 1.2000000000000002, fcut 0.9, xcut 0.0
